# Build APK cho game Thích Cầm Rìu trên Google Colab

Notebook này cài đặt **Java 17**, **Node.js 20**, **Android SDK** và dùng **Capacitor** để đóng gói thư mục `www/` thành file `.apk`.

### Chuẩn bị trên máy Windows trước khi chạy:
Chạy lệnh PowerShell sau trong thư mục dự án để tạo `throw-axe-colab.zip` (gọn nhẹ, không chứa `video/`):
```powershell
Compress-Archive -Path index.html,styles.css,manifest.webmanifest,package.json,capacitor.config.json,src,assets,vendor,scripts -DestinationPath throw-axe-colab.zip -Force
```

In [ ]:
# @title 1. Cài đặt Java 17, Node.js 20 và Android SDK
%%bash
set -e

echo "==> Cài Java 17 & unzip..."
apt-get update -qq
apt-get install -y -qq openjdk-17-jdk unzip wget curl > /dev/null

echo "==> Kiểm tra / cài Node.js 20..."
if ! node -v 2>/dev/null | grep -qE '^v(20|22|24|26)\.'; then
  curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null
  apt-get install -y -qq nodejs > /dev/null
fi
node -v
java -version

echo "==> Cài Android Command Line Tools..."
export ANDROID_HOME=/opt/android-sdk
if [ ! -d "$ANDROID_HOME/cmdline-tools/latest" ]; then
  mkdir -p "$ANDROID_HOME/cmdline-tools"
  wget -q https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip -O /tmp/cmdline-tools.zip
  unzip -q /tmp/cmdline-tools.zip -d "$ANDROID_HOME/cmdline-tools"
  mv "$ANDROID_HOME/cmdline-tools/cmdline-tools" "$ANDROID_HOME/cmdline-tools/latest"
  rm -f /tmp/cmdline-tools.zip
fi

yes | "$ANDROID_HOME/cmdline-tools/latest/bin/sdkmanager" --licenses > /dev/null || true
"$ANDROID_HOME/cmdline-tools/latest/bin/sdkmanager" "platform-tools" "platforms;android-34" "build-tools;34.0.0"
echo "==> Xong môi trường Android SDK!"

In [ ]:
# @title 2. Upload file throw-axe-colab.zip từ máy lên Colab
import os, shutil
from google.colab import files

WORK_DIR = '/content/throw-axe'
shutil.rmtree(WORK_DIR, ignore_errors=True)
os.makedirs(WORK_DIR, exist_ok=True)

uploaded = files.upload()
zip_name = next(iter(uploaded.keys()))
os.system(f'unzip -q -o "{zip_name}" -d "{WORK_DIR}"')
print('Đã giải nén vào:', WORK_DIR)
print('Nội dung:', os.listdir(WORK_DIR))

In [ ]:
# @title 3. Build gói web & đóng gói APK bằng Capacitor + Gradle
%%bash
set -e
export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64
export ANDROID_HOME=/opt/android-sdk
export ANDROID_SDK_ROOT=/opt/android-sdk
export PATH="$PATH:$ANDROID_HOME/cmdline-tools/latest/bin:$ANDROID_HOME/platform-tools"

cd /content/throw-axe
npm install --no-audit --no-fund @capacitor/core @capacitor/cli @capacitor/android
npm run build:web
if [ ! -d "android" ]; then
  npx cap add android
else
  npx cap sync android
fi

cd android
chmod +x gradlew
./gradlew assembleDebug

cp app/build/outputs/apk/debug/app-debug.apk /content/ThichCamRiu-debug.apk
ls -lh /content/ThichCamRiu-debug.apk

In [ ]:
# @title 4. Tải file APK về máy
from google.colab import files
files.download('/content/ThichCamRiu-debug.apk')